In [1]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np
from numpy.ma.core import less_equal
from tqdm import tqdm
import matplotlib.pyplot as plt

In [2]:
# Create Model
m = gp.Model("Model_1")

Set parameter Username
Set parameter LicenseID to value 2841584
Academic license - for non-commercial use only - expires 2027-07-06


In [3]:
# Time Based GMCNF parameters

# Gravitational acceleration [m/sˆ2]
g_0 = 9.80665

# Number of nodes i,j
nodes = 4

Connections = {0: [0,1] ,
               1: [0,1,2],
               2: [1,2,3],
               3: [2,3]}

# Time Steps 12 (days) #testing with +1 day
T = 12

# Advanced Time window
T_adv = list(range(T))
#print(T_adv)

#in case multiple time windows are needed this can be added

#Node Open Windows:
#Arcs can only depart or arrive at these nodes at the times specified (So always include the start and end of the window in the nodes)
#This makes every single arc unique based on its departing time and arrival node
#So: holding arc become multipliers for whatever time can be kept
N_Window = {0: [0,4,8,9,10,11],
             1:[0,5,9,10,11],
               2: [0,1,2,3,6,7,8,9,11],
                3:[0,2,3,4,5,6,11]}

reverseN_window = {key:list(reversed(item)) for key,item in N_Window.items()}


# Velocity change [km/StructureMass]
#this model optimizes for IMLEO,
#  so it does not consider any velocity chage necessary for LEO-PAC (splashdown)
#However, since the model includes splash down, and we want only a single launch per day,
#We can add a Large (big PropCapacity ) cost for PAC to LEO to ensure that the model does not use this arc
# unless it is really necessary

#Pacific Ocean, Low Earth Orbit, Lunar Lunar Orbit, Lunar Surface
# PAC, LEO, LLO, LS are 0, 1, 2, 3
delta_V = {0: {0: 0, 1: 1000}, # PAC to LEO is Big PropCapacity high
            1: {0: 0, 1: 0, 2: 4.04},
              2: {1: 4.04, 2: 0, 3: 1.87},
                3: {2: 1.87, 3: 0}}

# Time of travel [days]
TOF = {0: {0: 1, 1: 1},
        1: {0: 1, 1: 1, 2: 3},
          2: {1: 3, 2: 1, 3: 1},
            3: {2: 1, 3: 1}}

#TOF with the travel made backwards
ReverseTOF = {a:{} for a in TOF.keys()}
for a,b in TOF.items():
  for b1,c1 in b.items():
    ReverseTOF[a][b1] = -c1







#Only create variables for arcs that end within the destination window.

#TOF tells you the travel times
#def check_destination_window2(startnode, endnode, tstart, All_nodes= T_adv, TOF = TOF):
#    arrival =  TOF[startnode][endnode]+tstart

#    if arrival in All_nodes:
#        return True
#    else:  
#        return False



#Shows off all possible arcs
#In order [starttime][startnode][endnode]{"ArrivalTime", "FullTravelTime"}
def AllpossibleOutflowArcs(Connections, T_adv, window = N_Window, TOFused = TOF):
  

  AllArcs = {}

  for t in T_adv:
    TimeNode = {}
    for i in Connections:
      if t in window[i]:
        
        Now = window[i].index(t)
        TimeNode[i] = {}
        
        for j in Connections[i]:
            
            if t+TOFused[i][j] in window[j]:
              TimeNode[i][j]= {"ArrivalTime": t+TOFused[i][j], "FullTravelTime":TOFused[i][j] }

        #If the holding arc is not available, then the arc to the next available
        #free time block is added, as long as we are not at the end of the window (N_Window[i])
        if (i not in TimeNode[i]) and (Now+1 != len(window[i])): #remember Now is an index, starting at 0, len is the length of the window counting form 1
          
          TimeNode[i][i]={"ArrivalTime":window[i][Now+1],"FullTravelTime":window[i][Now+1] - t}
        
        
        #If Node is not at time 0: Previous Holding time:
        #print(i,Now)
        #if Now != 0:
        #  TimeNode[i][i]["PreviousHold"] = N_Window[i][Now-1]

        
        
    if TimeNode != {}:
      AllArcs[t] = TimeNode
            
                    
    
  return AllArcs
AllArcs = AllpossibleOutflowArcs(Connections,T_adv,window=N_Window,TOFused = TOF)
                
RevAllArcs = AllpossibleOutflowArcs(Connections,T_adv,window=reverseN_window,TOFused = ReverseTOF)
#print(AllArcs)
#print(RevAllArcs)
#print(RevAllArcs[1][2])
#print(RevAllArcs[2][2])
#print(RevAllArcs[2][3])


#for  k, v in AllArcs.items():
#   print(k)
#   print(v)
       


In [4]:
"""
#Simplified test data


# Number of vehicle types
V = 1



Y = GRB.INTEGER
# Spacecrafts of same type


# Structure mass [kg]
StructureMass = np.array([40000])

# Specific impulses [StructureMass]
I_sp = np.array([421])

# Payload Capacity [kg]
PayloadCap = np.array([5000])

# Propellant Capacity [kg]
PropCapacity = np.array([1200770])
"""


'\n#Simplified test data\n\n\n# Number of vehicle types\nV = 1\n\n\n\nY = GRB.INTEGER\n# Spacecrafts of same type\n\n\n# Structure mass [kg]\nStructureMass = np.array([40000])\n\n# Specific impulses [StructureMass]\nI_sp = np.array([421])\n\n# Payload Capacity [kg]\nPayloadCap = np.array([5000])\n\n# Propellant Capacity [kg]\nPropCapacity = np.array([1200770])\n'

In [5]:
#"""

# Number of vehicle types
V = 2



Y = GRB.INTEGER
# Spacecrafts of same type


#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]


# Structure mass [kg]
StructureMass = np.array([2500, 30000])

# Specific impulses [StructureMass]
I_sp = np.array([200, 500])

# Payload Capacity [kg]
PayloadCap = np.array([1000,75])

# Propellant Capacity [kg] 
PropCapacity = np.array([65000, 1700000])


#"""

In [6]:
"""
# VEHICLE DATA





# Number of vehicle types
V = 6



Y = GRB.INTEGER
# Spacecrafts of same type


#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]


# Structure mass [kg]
StructureMass = np.array([38415, 12014, 4841, 6053, 2770, 1719])

# Specific impulses [StructureMass]
I_sp = np.array([421, 421, 0, 314, 311, 311])

# Payload Capacity [kg]
PayloadCap = np.array([0, 0, 524, 60, 500, 250])

# Propellant Capacity [kg] 
PropCapacity = np.array([452045, 107725, 0, 18413, 8804, 2358])

"""







'\n# VEHICLE DATA\n\n\n\n\n\n# Number of vehicle types\nV = 6\n\n\n\nY = GRB.INTEGER\n# Spacecrafts of same type\n\n\n#Vehicles: [Saturn V stage 2, Saturn V stage 3, Command Module, Service Module, LM Descent Stage, LM ascent stage]\n\n\n# Structure mass [kg]\nStructureMass = np.array([38415, 12014, 4841, 6053, 2770, 1719])\n\n# Specific impulses [StructureMass]\nI_sp = np.array([421, 421, 0, 314, 311, 311])\n\n# Payload Capacity [kg]\nPayloadCap = np.array([0, 0, 524, 60, 500, 250])\n\n# Propellant Capacity [kg] \nPropCapacity = np.array([452045, 107725, 0, 18413, 8804, 2358])\n\n'

In [7]:
# COMMODITY Data and Demand/Supply

# Propellant mass fraction
#defined from rocket equation 1-e**(-deltav/Ispg0)

#actually the official function is e**(-deltav/Ispg0), but using the 1-e form allows use 
# to multiply the contents with the unchanging masses to include their input into the transformation linearly
#This varies with the delta v necessary for each arc, and the Isp of the vehicle used for that arc
#Will be used later


def phi(i,j,v, dV = delta_V, I_sp = I_sp, g_0 = g_0):
    if I_sp[v] == 0:
        return 1
    else:
        return 1 - np.exp(-(1000*dV[i][j] / (I_sp[v] * g_0))) #1000 used for conversion



# Commodity variable types
# Crew, consumables kg, equipment kg, samples kg, propellant kg
X = [GRB.INTEGER, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS, GRB.CONTINUOUS]
PropIndex = 4

# Crew mass [kg/crew]
crew_mass = 100


CommodityMassConversion = [crew_mass,1,1,1,1]

# Crew, consumables kg, equipment kg, samples kg, propellant kg
# PAYLOAD ASSUMPTIONS

# Consumption rates [kg/crew/day]
food_consumption = 1.0
water_consumption = 5.0
oxygen_consumption = 1.1
consumption = food_consumption + water_consumption + oxygen_consumption
#if the model works, this can be made more granular by separating the consumptions



# Upper limit of each variable per spacecraft is the capacity of each spacecraft *number of spacecraft in that node.
# Crew, consumables kg, equipment kg, samples kg, propellant kg
XUpper = [PayloadCap/crew_mass, PayloadCap, PayloadCap, PayloadCap, PropCapacity]

#Single Spacecraft consumption table (from outflow to inflow consumption of all commodities)

#Matrix multiplication with a vector of outflows


# There is no propellant output
def Solo_SC_Consumption(i, j,v, consumption = consumption, TOF = TOF,structure_mass = StructureMass):

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft
    value = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [crew_mass * -1*phi(i,j,v,delta_V,I_sp,g_0),
                       -1*phi(i,j,v,delta_V,I_sp,g_0),
                         -1*phi(i,j,v,delta_V,I_sp,g_0),
                           -1*phi(i,j,v,delta_V,I_sp,g_0),
                             1-1*phi(i,j,v,delta_V,I_sp,g_0),
                                -1*structure_mass[v]*phi(i,j,v,delta_V,I_sp,g_0)], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    return value

def Solo_SC_Consumption_NodV(i, j,v, consumption = consumption, TOF = TOF,structure_mass = StructureMass):

    # Crew, consumables kg, equipment kg, samples kg, propellant kg, number of spacecraft
    value = np.array([[1, 0, 0, 0, 0, 0], #Crew
                        [-consumption*TOF[i][j] , 1, 0, 0, 0, 0], # Consumable consumption
                        [0, 0, 1, 0, 0, 0], #Equipment
                        [0, 0, 0, 1, 0, 0], #Samples
                        [0, 0, 0, 0, 1, 0], #Propellant
                        [0, 0, 0, 0, 0, 1]])# last row (and column) is for number of spacecraft
    
    return value








# THE COMMODITIES ARE PROVIDED AT LEO. START ALL THINGS AT LEO
# CHECK DAYS FOR MISSION !!!
#Commodity Array: D[Node][Day][Commodity]



D = [[np.array([0 for x in range(len(X))])
      for _ in T_adv]
    for _ in Connections]

print(len(D))
print(len(D[0]))
print(len(D[0][0]))

#initial test values
# Earth (PAC) consumables, equipment, propellant supply infinite at leo time 0, AND Moon surface sample supply (infinite at all times)
#Crew are capped in supply so we don't leave anyone on the moon
D[1][0][0] = 3 #Crew
D[1][0][1] = 99999 #Consumables
D[1][0][2] = 99999 #Equipment
D[1][0][4] = 99999999999 #Propellant

for x in T_adv:
    D[3][x][3] = 999999 #Moon samples


# APOLLO
# Remember we are using a list starting at 0
# Crew demand/supply
D[3][4][0] = -2 # Lunar surface day 5 crew demand (negative supply)
D[2][3][0] = -1 # Lunar orbit day 4 crew demand
D[3][5][0] = 2 # Lunar surface day 6 crew supply (return)
D[2][6][0] = 1 # Lunar orbit day 7 crew supply (return)
D[0][11][0] = -3 # Earth day 11 crew demand (return)

D[3][4][2] = -420 # Lunar surface day 5 (scientific) equipment demand

D[0][11][3] = -110 # Earth day 11 lunar sample demand





# S/PayloadCap COMMODITY DEMAND
# format d[node][vehicle][day]
# Infinite supply of SC at LEO day 1 (time 0), none elsewhere
d = [[[1 if (i == 1 and t == 0) else 0 for t in range(T)] # Infinite supply of spacecrafts at i = 1, t = 0 LEO
     for _ in range(V)]
    for i in Connections]


4
12
5


In [8]:
# CREATE COMMODITY FLOW VECTORS AND S/PayloadCap COMMODITY FLOW
# Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}'
#v: vehicle type, i: node of origin, j: node of destination, t: time step, x: commodity type
# Spacecraft Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}'




#Only create variables for arcs that end within the destination window.

#TOF tells you the travel times
def check_destination_window(startnode, endnode, tstart, All_nodes= T_adv, TOF = TOF):
    arrival =  TOF[startnode][endnode]+tstart

    if arrival in All_nodes:
        return True
    else:
        return False







#lower bound for all commodities is 0, no negatives.
def create_commodity_flow(model, V, X, Time = T_adv, direction = "out", connect = Connections, typeC = "Classic"):

    
    
    #x_flow = [[{j: [np.array([[model.addVar(vtype=X[x], name=f'{typeC}_commodity_{direction}flow_{v},{i},{j},{t},{x}',lb = 0 )]
    #                          for x in range(len(X))])
    #                for t in range(T-1) if check_destination_window(i, j, t, Time, TOF)]
    #            for j in connect[i]}
    #           for i in connect ]
    #          for v in range(V)]




    x_flow = {v:{i:{j: {t: np.array([[model.addVar(vtype=X[x], name=f'{typeC}_commodity_{direction}flow_{v},{i},{j},Tstart{t},Tend{AllArcs[t][i][j]['ArrivalTime']},Commodity{x}',lb = 0 )]
                              for x in range(len(X))])
                    for t in AllArcs if (i in AllArcs[t]) and (j in AllArcs[t][i])} #Only make an arc if it exists in AllArcs
                for j in connect[i]}
               for i in connect}
              for v in range(V)}


    return x_flow


def create_sc_commodity_flow(model, V,Y, Time = T_adv, direction = "out", connect = Connections):
    y_flow = {v:{i:{j: {t: np.array([model.addVar(vtype=Y, name=f'sc_commodity_{direction}flow_{v},{i},{j},Tstart{t},Tend{AllArcs[t][i][j]['ArrivalTime']}',lb=0)])
            for t in AllArcs if (i in AllArcs[t]) and (j in AllArcs[t][i])}
          for j in connect[i]}
         for i in connect} for v in range(V)}

    return y_flow

# Outflow+ leaving from node i to j, inflow- arriving at node j from i

x_outflow, x_inflow = create_commodity_flow(m, V, X, T_adv, direction="out", connect=Connections), create_commodity_flow(m, V, X, T_adv, direction="in", connect=Connections)
y_outflow, y_inflow = create_sc_commodity_flow(m, V, Y, T_adv, direction="out", connect=Connections), create_sc_commodity_flow(m, V, Y, T_adv, direction="in", connect=Connections)






m.update()


print(AllArcs[6][3])
print(len(y_outflow[0][0][0]))

print(y_outflow[0][0][0][4])
print(x_outflow[0][0][0][4])
print(x_outflow[1][3][3][6])





{2: {'ArrivalTime': 7, 'FullTravelTime': 1}, 3: {'ArrivalTime': 11, 'FullTravelTime': 5}}
5
[<gurobi.Var sc_commodity_outflow_0,0,0,Tstart4,Tend8>]
[[<gurobi.Var Classic_commodity_outflow_0,0,0,Tstart4,Tend8,Commodity0>]
 [<gurobi.Var Classic_commodity_outflow_0,0,0,Tstart4,Tend8,Commodity1>]
 [<gurobi.Var Classic_commodity_outflow_0,0,0,Tstart4,Tend8,Commodity2>]
 [<gurobi.Var Classic_commodity_outflow_0,0,0,Tstart4,Tend8,Commodity3>]
 [<gurobi.Var Classic_commodity_outflow_0,0,0,Tstart4,Tend8,Commodity4>]]
[[<gurobi.Var Classic_commodity_outflow_1,3,3,Tstart6,Tend11,Commodity0>]
 [<gurobi.Var Classic_commodity_outflow_1,3,3,Tstart6,Tend11,Commodity1>]
 [<gurobi.Var Classic_commodity_outflow_1,3,3,Tstart6,Tend11,Commodity2>]
 [<gurobi.Var Classic_commodity_outflow_1,3,3,Tstart6,Tend11,Commodity3>]
 [<gurobi.Var Classic_commodity_outflow_1,3,3,Tstart6,Tend11,Commodity4>]]


In [9]:
# ADD THE CONSTRAINTS (2 & 3)
# CONSTRAINTS 2 & 3 MASS BALANCE
# Node commodity demand D vectors (positive for supply)
# sum(x[i][t]+) - sum(x[i][t]-) <= D[i][t]

import sys


#Outflow is based off of of the AllArcs output i
#Inflow is based off of the AllArcs input j, so Based off of ArrivalTime
#TOF still applies, but with the extra window check (or both work with AllArcs)


#Remember: And logic is sequential, only the first failure stops it,
# so logical statements based on previous true statements can be added after each other

for t in AllArcs:
    for i in AllArcs[t]:

#for i in Connections:
#    for t in T_adv: #range(T - 1)) also works for simple tests


        print(t,i)

        x_outflow_sum = sum(x_outflow[v][i][j][t] 
                            if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i])
                            else np.array([[0] for _ in range(len(X))])  #packaged SC are dealt with differently
                            for v in range(V) 
                            for j in Connections[i])
        
        
        
        
        x_inflow_sum = sum(x_inflow[v][j][i][RevAllArcs[t][i][j]["ArrivalTime"]] 
                           if (t in RevAllArcs) and (i in RevAllArcs[t]) and (j in RevAllArcs[t][i])
                           else np.array([[0] for _ in range(len(X))])
                           for v in range(V) 
                           for j in Connections[i])
        # Only count the inflows for which the spacecraft has had time to arrive 
        # (or where it has had time to depart (no negative times))
        #And make sure it is coming from a window that is open 
        #We are not using revallarcs to pick the j values, since we need x inflow sum to have array 
        

        

        for x in range(len(X)):
            if (t in AllArcs) and (i in AllArcs[t]):
                try:
                    print(x_outflow_sum)
                    print(x_inflow_sum)
                    
                    m.addConstr(x_outflow_sum[x][0] - x_inflow_sum[x][0] <= D[i][t][x],
                                name=f"mass_balance_x_node{i}_time{t}_comm{x}")
                    #print(x_outflow_sum[x][0])
                
                except Exception as e:
                    print(f"Error on node={i}, time={t}, commodity={x}: {e}")
                    raise
            
            

        # S/C commodity supply and demand
        for v in range(V):
            y_outflow_sum = sum(y_outflow[v][i][j][t]  
                                if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i])\
                                else np.array([0]) 
                                for j in Connections[i])

            y_inflow_sum = sum(y_inflow[v][j][i][RevAllArcs[t][i][j]["ArrivalTime"]] 
                               if (t in RevAllArcs) and (i in RevAllArcs[t]) and (j in RevAllArcs[t][i])\
                               else np.array([0])
                               for j in Connections[i])

                

            #temporary pick and choose, change this later, only sinks at the end of the model

            if (t in AllArcs) and (i in AllArcs[t]):

                if (t < T-1) and (t != 0):

                    m.addConstr(y_outflow_sum[0] - y_inflow_sum[0] == d[i][v][t],
                    name=f"SC_lossless_mass_balance_x_node{i}_time{t}_vehicle{v}") #lose no SC until the end
                else:
                    m.addConstr(y_outflow_sum[0] - y_inflow_sum[0] <= d[i][v][t],
                    name=f"SC_mass_balance_x_node{i}_time{t}_vehicle{v}")


        
        


m.update()

0 0
[[<gurobi.LinExpr: Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity0 + Classic_commodity_outflow_1,0,0,Tstart0,Tend4,Commodity0>]
 [<gurobi.LinExpr: Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity1 + Classic_commodity_outflow_1,0,0,Tstart0,Tend4,Commodity1>]
 [<gurobi.LinExpr: Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity2 + Classic_commodity_outflow_1,0,0,Tstart0,Tend4,Commodity2>]
 [<gurobi.LinExpr: Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity3 + Classic_commodity_outflow_1,0,0,Tstart0,Tend4,Commodity3>]
 [<gurobi.LinExpr: Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity4 + Classic_commodity_outflow_1,0,0,Tstart0,Tend4,Commodity4>]]
[[0]
 [0]
 [0]
 [0]
 [0]]
[[<gurobi.LinExpr: Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity0 + Classic_commodity_outflow_1,0,0,Tstart0,Tend4,Commodity0>]
 [<gurobi.LinExpr: Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity1 + Classic_commodity_outflow_1,0,0,Tstart0,Tend4,Commodity1>]
 [<gurobi.

In [10]:
## Commodity transformation
#import sys



for t in AllArcs:
    for i in AllArcs[t]:
        for j in AllArcs[t][i]:
#for i in Connections:
#    for j in Connections[i]:
#        for t in T_adv:

             #if check_destination_window(i,j,t,T_adv,TOF):
             #if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i]): #Make sure there is an arc to transform
                  
                for v in range(V):
                
                
                    #print(i,j,t,v)
                
                
                    Vout = np.concatenate((x_outflow[v][i][j][t],
                        np.array([y_outflow[v][i][j][t]])), axis=0)


                    Vin = np.concatenate((x_inflow[v][i][j][t],
                         np.array([y_inflow[v][i][j][t]])), axis=0)
                    




                    #print(Vin)
                    #sys.exit()


                    #create the correct consumption matrix, based on deltav and travel time
                    if delta_V[i][j] == 0: #If there is no Delta v: there is no propellant consumption
                        Consumed =Solo_SC_Consumption_NodV(i,j,v,consumption,TOF,StructureMass)
                    
                    else:
                        Consumed =Solo_SC_Consumption(i,j,v,consumption,TOF,StructureMass)
                    
                    

                    


                    transformed = np.dot(Consumed,Vout)
                
                    
                    for i1,(enterarc,leavearc) in enumerate(zip(transformed, Vin)):

                        m.addConstr(enterarc[0] == leavearc[0],name=f'Arc_transformationConstraint_Start{i}_End{j}_Starttime{t}_Vehicle{v}_Commodity{i1}')

In [11]:
# CONSTRAINTS 5 CONCURRENCY LIMITS

# Concurrency constraint matrix
# H[x+] <= e * y+ --> Payload mass and fuel in StructureMass/c does not exceed maximum capacities
# x = Crew, consumables, equipment, samples, propellant

# def create_concurrency_constraint(V, connect =Connections, crew_mass): # Different vehicles version
#     H = [[{j: np.array([[crew_mass, 1, 1, 1, 0],
#                         [0, 0, 0, 0, 1]])
#            for j in connect[i]}
#           for i in connect]
#          for v in range(V)]
#
#     return H


def create_concurrency_constraint(connect = Connections, crew_mass = crew_mass): # Same for all vehicles, max payload mass
    H = [{j: np.array([[crew_mass, 1, 1, 1, 0], #payload
                        [0, 0, 0, 0, 1]]) #Propellant
           for j in connect[i]}
          for i in connect]
    return H


def create_sc_design_parameters(V, PayloadCap, PropCapacity):
    e = [np.array([[PayloadCap[v]], [PropCapacity[v]]]) for v in range(V)]
    return e


H = create_concurrency_constraint(Connections, crew_mass)
e = create_sc_design_parameters(V, PayloadCap, PropCapacity)
print(H)
print(len(H))

[{0: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 1: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]])}, {0: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 1: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 2: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]])}, {1: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 2: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 3: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]])}, {2: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]]), 3: array([[100,   1,   1,   1,   0],
       [  0,   0,   0,   0,   1]])}]
4


In [12]:
# ADD THE CONSTRAINTS (5)

for v in range(V):

    for t in AllArcs:
        for i in AllArcs[t]:
            for j in AllArcs[t][i]:
    #for i in Connections:
    #    for j in Connections[i]:
    #        for t in range(T-1):
                #if check_destination_window(i,j,t,T_adv,TOF):
    #            if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i]):

                    for commodity, constraint in zip(np.dot(H[i][j], x_outflow[v][i][j][t]),
                                                     e[v]*y_outflow[v][i][j][t][0]):

                        m.addConstr(commodity[0] <= constraint[0])

m.update()


In [13]:
# CONSTRAINTS 6 TIME-WINDOW
# ADD THE CONSTRAINTS (6)

#Minimum value of 0 for all arcs

for v in range(V):
    for i in Connections:
        for j in Connections[i]:
            for t in range(T-1):
                #if check_destination_window(i,j,t,T_adv,TOF):
                if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i]):

                    for commodity_out in x_outflow[v][i][j][t]:
                        m.addConstr(commodity_out[0] >= 0)

                    for commodity_in in x_inflow[v][i][j][t]:
                        m.addConstr(commodity_in[0] >= 0)

                    m.addConstr(y_outflow[v][i][j][t][0] >= 0)
                    m.addConstr(y_inflow[v][i][j][t][0] >= 0)

m.update()

# StructureMass[v] >= 0


In [14]:
# CONSTRAINTS 7 SPACE-CRAFT MASS

In [15]:
# COST FUNCTION - INITIAL MASS AT LEO
# sum(cost * x + cost_y * StructureMass * y)

# x = Crew, consumables, equipment, samples, propellant, crew(return)

#Currently the model assumes we are starting at the LEO node t = 0, i=1

def create_commodity_cost(V, connect = Connections, crew_mass= crew_mass):
    cost_coeff = [[{j:
                        [np.array([[crew_mass], [1], [1], [1], [1]]) if (t == 0 and i == 1)
                         else np.array([[0] for _ in range(len(X))])
                         for t in range(T-1) if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i])]
           for j in connect[i]}
          for i in connect]
         for _ in range(V)]

    sc_cost_coeff = [[{j:
                        [1 if (t == 0 and i == 1)
                         else 0
                         for t in range(T-1) if (t in AllArcs) and (i in AllArcs[t]) and (j in AllArcs[t][i])]
           for j in connect[i]}
          for i in connect]
         for _ in range(V)]

    return cost_coeff, sc_cost_coeff

cost_coeff, sc_cost_coeff = create_commodity_cost(V, Connections, crew_mass)

In [16]:
# DEFINE THE COST FUNCTION (1)

"""
#General version
cost = sum(
    np.dot(cost_coeff[v][i][j][t].T, x_outflow[v][i][j][t]) + sc_cost_coeff[v][i][j][t] * StructureMass[v] * y_outflow[v][i][j][t][0]
    for v in range(V)
    for i in Connections
    for j in Connections[i]
    for t in range(3)
)
"""
#specific to apollo version
#only looking at node 1 at time t = 0
cost = sum(
    np.dot(cost_coeff[v][1][j][0].T, x_outflow[v][1][j][0]) + sc_cost_coeff[v][1][j][0] * StructureMass[v] * y_outflow[v][1][j][0][0]
    for v in range(V)
    for j in Connections[1] if (0 in AllArcs) and (1 in AllArcs[0]) and (j in AllArcs[0][1])
)

cost = cost[0][0]
print(cost)

m.setObjective(cost, GRB.MINIMIZE)
m.update()

100.0 Classic_commodity_outflow_0,1,1,Tstart0,Tend5,Commodity0 + Classic_commodity_outflow_0,1,1,Tstart0,Tend5,Commodity1 + Classic_commodity_outflow_0,1,1,Tstart0,Tend5,Commodity2 + Classic_commodity_outflow_0,1,1,Tstart0,Tend5,Commodity3 + Classic_commodity_outflow_0,1,1,Tstart0,Tend5,Commodity4 + 2500.0 sc_commodity_outflow_0,1,1,Tstart0,Tend5 + 100.0 Classic_commodity_outflow_0,1,2,Tstart0,Tend3,Commodity0 + Classic_commodity_outflow_0,1,2,Tstart0,Tend3,Commodity1 + Classic_commodity_outflow_0,1,2,Tstart0,Tend3,Commodity2 + Classic_commodity_outflow_0,1,2,Tstart0,Tend3,Commodity3 + Classic_commodity_outflow_0,1,2,Tstart0,Tend3,Commodity4 + 2500.0 sc_commodity_outflow_0,1,2,Tstart0,Tend3 + 100.0 Classic_commodity_outflow_1,1,1,Tstart0,Tend5,Commodity0 + Classic_commodity_outflow_1,1,1,Tstart0,Tend5,Commodity1 + Classic_commodity_outflow_1,1,1,Tstart0,Tend5,Commodity2 + Classic_commodity_outflow_1,1,1,Tstart0,Tend5,Commodity3 + Classic_commodity_outflow_1,1,1,Tstart0,Tend5,Commodity4

In [17]:
m.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1869 rows, 1008 columns and 3858 nonzeros (Min)
Model fingerprint: 0x7a4e6d34
Model has 24 linear objective coefficients
Variable types: 672 continuous, 336 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-01, 2e+06]
  Objective range  [1e+00, 3e+04]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+11]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.

Presolve removed 1806 rows and 939 columns
Presolve time: 0.02s
Presolved: 63 rows, 69 columns, 250 nonzeros
Variable types: 57 continuous, 12 integer (9 binary)
Found heuristic solution: objective 266879.32219
Found heuristic solution: objective 266547.00187

Root relaxation: objective 1.685738e+05, 45 iterations, 0.00 seconds (0.00 work units)

  

In [18]:

"""
# Good practice: write the full model first
m.update()
m.write("debug_model.lp")

m.optimize()

# If Gurobi says infeasible or unbounded, resolve with DualReductions off
if m.Status == GRB.INF_OR_UNBD:
    m.Params.DualReductions = 0
    m.optimize()

if m.Status == GRB.INFEASIBLE:
    print("Model is infeasible. Computing IIS...")
    m.computeIIS()

    # Writes a small model containing the infeasible subsystem
    m.write("infeasible_subset.ilp")

    print("\nConstraints in IIS:")
    for c in m.getConstrs():
        if c.IISConstr:
            print(f"{c.ConstrName}: sense={c.Sense}, RHS={c.RHS}")

    print("\nVariable bounds in IIS:")
    for v in m.getVars():
        if v.IISLB:
            print(f"{v.VarName}: lower bound {v.LB}")
        if v.IISUB:
            print(f"{v.VarName}: upper bound {v.UB}")

"""

'\n# Good practice: write the full model first\nm.update()\nm.write("debug_model.lp")\n\nm.optimize()\n\n# If Gurobi says infeasible or unbounded, resolve with DualReductions off\nif m.Status == GRB.INF_OR_UNBD:\n    m.Params.DualReductions = 0\n    m.optimize()\n\nif m.Status == GRB.INFEASIBLE:\n    print("Model is infeasible. Computing IIS...")\n    m.computeIIS()\n\n    # Writes a small model containing the infeasible subsystem\n    m.write("infeasible_subset.ilp")\n\n    print("\nConstraints in IIS:")\n    for c in m.getConstrs():\n        if c.IISConstr:\n            print(f"{c.ConstrName}: sense={c.Sense}, RHS={c.RHS}")\n\n    print("\nVariable bounds in IIS:")\n    for v in m.getVars():\n        if v.IISLB:\n            print(f"{v.VarName}: lower bound {v.LB}")\n        if v.IISUB:\n            print(f"{v.VarName}: upper bound {v.UB}")\n\n'

In [19]:
import sys
sys.path.append("/2026_code/")  # needed to add in the results file
import Results

m.update()
m.write("debug_model_Simple.lp")

Apollo_Veh =[
        "Saturn V S-II",
        "Saturn V S-IVB",
        "Command Module",
        "Service Module",
        "LM Descent Stage",
        "LM Ascent Stage"
    ]

Two_ship_test =["Payload","Fuel"]

namevar = Two_ship_test

Cargoflows,Shipflows = Results.extract_flows(
    x_outflow=x_outflow,
    x_inflow=x_inflow,
    y_outflow=y_outflow,
    arcs=Connections,
    node_names=["PAC", "LEO", "LLO", "LS"],
    vehicle_names=namevar,
    commodity_names=[
        "crew",
        "consumables",
        "equipment",
        "sample",
        "propellant"
    ],
    tof=TOF,
    T=T,
    T_adv = T_adv, 
    CommMass = CommodityMassConversion,
    AllArcs=AllArcs,
    StructMass= StructureMass
)

#flows.head()


Results.plot_time_space_network(
    Shipflows,
    Cargoflows,
    node_order=["PAC", "LEO", "LLO", "LS"],
    title="Apollo 17 optimized time-space logistics solution"
)

mass_table = Results.make_mass_flow_table(Cargoflows, use="out_mass")
#mass_table

mass_table.style.format(precision=1)

Results.propellantUsage(Cargoflows)

Results.plot_vehicle_gantt(Cargoflows, title="Apollo 17 spacecraft activity")

In [20]:
mass_table.style.format(precision=1)

item,t_depart,t_arrive,from_node,to_node,vehicle,n_ships,consumables,crew,equipment,propellant,sample,total_flow
0,0,3,LEO,LLO,Fuel,1.0,0.0,0.0,75.0,123700.2,0.0,123775.2
1,0,3,LEO,LLO,Payload,1.0,198.8,300.0,345.0,22887.0,0.0,23730.8
3,3,6,LLO,LLO,Fuel,1.0,75.0,0.0,0.0,20501.0,0.0,20576.0
2,3,4,LLO,LS,Payload,1.0,59.9,200.0,420.0,16885.8,0.0,17565.7
4,4,5,LS,LS,Payload,1.0,45.7,0.0,0.0,4553.7,0.0,4599.4
5,5,6,LS,LLO,Payload,1.0,45.7,200.0,0.0,4553.7,110.0,4909.4
6,6,7,LLO,LLO,Fuel,1.0,0.0,0.0,0.0,0.0,75.0,75.0
7,6,7,LLO,LLO,Payload,1.0,106.5,300.0,0.0,20501.0,35.0,20942.5
8,7,10,LLO,LEO,Payload,1.0,85.2,300.0,0.0,20501.0,110.0,20996.2
9,10,11,LEO,PAC,Payload,1.0,21.3,300.0,0.0,0.0,110.0,431.3


In [21]:
Shipflows.head()

,vehicle,v,from_node,to_node,i,j,t_depart,t_arrive,active_vehicle_mass,cargo_mass,total_outgoing_mass,n_ships
0,Payload,0,LEO,PAC,1,0,10,11,2500.0,431.300000,2931.300000,1.0
1,Payload,0,LEO,LLO,1,2,0,3,2500.0,23730.808976,26230.808976,1.0
2,Payload,0,LLO,LEO,2,1,7,10,2500.0,20996.177715,23496.177715,1.0
3,Payload,0,LLO,LLO,2,2,6,7,2500.0,20942.477715,23442.477715,1.0
4,Payload,0,LLO,LS,2,3,3,4,2500.0,17565.686785,20065.686785,1.0


In [22]:
# Print the values of all variables
for v in m.getVars():
    print(f"{v.VarName} = {v.X}")

Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity0 = 0.0
Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity1 = 0.0
Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity2 = 0.0
Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity3 = 0.0
Classic_commodity_outflow_0,0,0,Tstart0,Tend4,Commodity4 = 0.0
Classic_commodity_outflow_0,0,0,Tstart4,Tend8,Commodity0 = 0.0
Classic_commodity_outflow_0,0,0,Tstart4,Tend8,Commodity1 = 0.0
Classic_commodity_outflow_0,0,0,Tstart4,Tend8,Commodity2 = 0.0
Classic_commodity_outflow_0,0,0,Tstart4,Tend8,Commodity3 = 0.0
Classic_commodity_outflow_0,0,0,Tstart4,Tend8,Commodity4 = 0.0
Classic_commodity_outflow_0,0,0,Tstart8,Tend9,Commodity0 = 0.0
Classic_commodity_outflow_0,0,0,Tstart8,Tend9,Commodity1 = 0.0
Classic_commodity_outflow_0,0,0,Tstart8,Tend9,Commodity2 = 0.0
Classic_commodity_outflow_0,0,0,Tstart8,Tend9,Commodity3 = 0.0
Classic_commodity_outflow_0,0,0,Tstart8,Tend9,Commodity4 = 0.0
Classic_commodity_outflow_0,0,0,Tstart9,Tend10,Commodit

In [23]:
"""

# x = Crew, consumables, equipment, samples, propellant
#Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}'
# Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}'
import pandas as pd

results = {final_variable.VarName: final_variable.X for final_variable in m.getVars()}

sorted_results = dict(sorted(results.items(), key=lambda item: int(item[0].split(",")[3])))

f = open("MostBasic.txt", "w")



for final in sorted_results:
    if sorted_results[final] != 0:
        print('%StructureMass %g' % (final, sorted_results[final]))
        f.write('%StructureMass %g' % (final, sorted_results[final]))
        f.write('\n')
f.close()
"""

'\n\n# x = Crew, consumables, equipment, samples, propellant\n#Variable naming process: commodity_{direction}flow_{v},{i},{j},{t},{x}\'\n# Variable naming process: sc_commodity_{direction}flow_{v},{i},{j},{t}\'\nimport pandas as pd\n\nresults = {final_variable.VarName: final_variable.X for final_variable in m.getVars()}\n\nsorted_results = dict(sorted(results.items(), key=lambda item: int(item[0].split(",")[3])))\n\nf = open("MostBasic.txt", "w")\n\n\n\nfor final in sorted_results:\n    if sorted_results[final] != 0:\n        print(\'%StructureMass %g\' % (final, sorted_results[final]))\n        f.write(\'%StructureMass %g\' % (final, sorted_results[final]))\n        f.write(\'\n\')\nf.close()\n'

In [24]:
# # EQUATION 7 CONSTRAINTS
#
# # Structural Fraction (fuel dependent)
# alpha = 0.045  # LOX/kerosene
#
# # Gravitational Acceleration Earth
# g_0 = 9.8  # m/s2
#
# # Upper Bound Allowed for Propellant Tank Capacity
# M_ub = 500000  # kg
#
# # Spacecraft Impulsive Burn
# t_b = 120  # StructureMass
#
#
# # Structure Mass Variable
# def create_s_star_variables(model, v=V):
#     variables = {}
#     for v in range(V):
#         variables[v] = model.addVar(vtype=GRB.CONTINUOUS, name=f'Structure_Mass_{v}')
#     return variables
#
#
# s_star = create_s_star_variables(model=m)
#
# m.update()

In [25]:
# # CONSTRAINTS 7
#
# for v in tqdm(V):
#     m.addConstr(s_star[v] = 2.3931 * )